In [1]:
"""
DATASET DIAGNOSTIC & FIX SCRIPT
Run this BEFORE training to check and fix your dataset

This will:
1. Find corrupted/unreadable images
2. Check image formats
3. Validate all images can be loaded
4. Remove or fix problematic files
"""

import os
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import shutil

# ============================================
# CONFIGURATION
# ============================================
DATASET_PATH = "/kaggle/input/speedlimit"  # Change if needed
WORK_DIR = "/kaggle/working"

print("="*60)
print("DATASET DIAGNOSTIC TOOL")
print("="*60)

# ============================================
# FIND ALL IMAGES
# ============================================
def find_all_images(path):
    """Find all image files in dataset"""
    extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG']
    images = []
    
    for ext in extensions:
        images.extend(Path(path).rglob(f'**/*{ext}'))
    
    return images

print(f"\n🔍 Scanning dataset: {DATASET_PATH}")
all_images = find_all_images(DATASET_PATH)
print(f"✓ Found {len(all_images)} images")

# ============================================
# TEST IMAGE LOADING
# ============================================
def test_image_loading(img_path):
    """Test if image can be loaded with OpenCV and PIL"""
    errors = []
    
    # Test OpenCV
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            errors.append("OpenCV: imread returned None")
    except Exception as e:
        errors.append(f"OpenCV: {str(e)}")
    
    # Test PIL
    try:
        img = Image.open(img_path)
        img.verify()
    except Exception as e:
        errors.append(f"PIL: {str(e)}")
    
    return errors

print("\n🧪 Testing image loading...")
problematic_images = []

for i, img_path in enumerate(all_images):
    if i % 100 == 0:
        print(f"   Tested {i}/{len(all_images)} images...", end='\r')
    
    errors = test_image_loading(img_path)
    if errors:
        problematic_images.append({
            'path': img_path,
            'errors': errors
        })

print(f"   Tested {len(all_images)}/{len(all_images)} images    ")

# ============================================
# REPORT RESULTS
# ============================================
print(f"\n📊 DIAGNOSTIC RESULTS:")
print("="*60)

if not problematic_images:
    print("✅ All images are OK!")
else:
    print(f"⚠️  Found {len(problematic_images)} problematic images:")
    print()
    
    for item in problematic_images[:10]:  # Show first 10
        print(f"❌ {item['path'].name}")
        for error in item['errors']:
            print(f"   - {error}")
        print()
    
    if len(problematic_images) > 10:
        print(f"   ... and {len(problematic_images) - 10} more")

# ============================================
# FIX STRATEGY
# ============================================
if problematic_images:
    print("\n🔧 FIX STRATEGY:")
    print("="*60)
    print("""
Option 1: Copy dataset and remove bad images (RECOMMENDED)
Option 2: Try to convert/fix bad images
Option 3: Use cache=False in training
    """)
    
    # Create clean dataset
    print("\n📦 Creating clean dataset...")
    clean_dataset = f"{WORK_DIR}/speedlimit_clean"
    
    # Copy dataset structure
    for split in ['train', 'valid', 'val']:
        for subdir in ['images', 'labels']:
            os.makedirs(f"{clean_dataset}/{split}/{subdir}", exist_ok=True)
    
    # Get problematic image paths as set
    bad_paths = {str(item['path']) for item in problematic_images}
    
    # Copy good images and their labels
    copied = 0
    skipped = 0
    
    for img_path in all_images:
        if str(img_path) not in bad_paths:
            # Determine split (train/valid/val)
            parts = img_path.parts
            split = None
            for part in parts:
                if part in ['train', 'valid', 'val']:
                    split = part
                    break
            
            if split:
                # Copy image
                dest_img = f"{clean_dataset}/{split}/images/{img_path.name}"
                shutil.copy2(str(img_path), dest_img)
                
                # Copy corresponding label
                label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
                if label_path.exists():
                    dest_lbl = f"{clean_dataset}/{split}/labels/{img_path.stem}.txt"
                    shutil.copy2(str(label_path), dest_lbl)
                
                copied += 1
        else:
            skipped += 1
    
    print(f"✓ Copied {copied} good images")
    print(f"✓ Skipped {skipped} bad images")
    
    # Create data.yaml for clean dataset
    import yaml
    
    # Try to read original classes
    original_yaml = list(Path(DATASET_PATH).rglob('*.yaml'))
    classes = None
    
    if original_yaml:
        with open(original_yaml[0]) as f:
            data = yaml.safe_load(f)
            classes = data.get('names', None)
    
    if not classes:
        classes = ['speed_limit_100', 'speed_limit_120', 'speed_limit_20', 
                  'speed_limit_30', 'speed_limit_40', 'speed_limit_50', 
                  'speed_limit_60', 'speed_limit_70', 'speed_limit_80', 
                  'speed_limit_90', 'stop']
    
    clean_yaml = {
        'path': clean_dataset,
        'train': 'train/images',
        'val': 'valid/images',
        'nc': len(classes),
        'names': classes
    }
    
    yaml_path = f"{clean_dataset}/data.yaml"
    with open(yaml_path, 'w') as f:
        yaml.dump(clean_yaml, f, default_flow_style=False)
    
    print(f"✓ Created data.yaml: {yaml_path}")
    print(f"\n✅ Clean dataset ready at: {clean_dataset}")
    print(f"\n📝 UPDATE YOUR TRAINING CODE:")
    print(f"   data_yaml = '{yaml_path}'")

# ============================================
# CHECK SEGMENT/BOX WARNING
# ============================================
print("\n\n🔍 CHECKING FOR SEGMENT ISSUES:")
print("="*60)

# Check label files for segment data
label_files = list(Path(DATASET_PATH).rglob('**/labels/*.txt'))
segment_issues = 0

for lbl_file in label_files[:100]:  # Check first 100
    try:
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                # If more than 5 values, it might be a segment
                if len(parts) > 5:
                    segment_issues += 1
                    break
    except:
        pass

if segment_issues > 0:
    print(f"⚠️  Found {segment_issues} labels with segmentation data")
    print("   This can cause warnings but usually doesn't break training")
    print("   YOLO will automatically use only bounding boxes")
else:
    print("✅ No segmentation issues detected")

# ============================================
# SUMMARY
# ============================================
print("\n\n" + "="*60)
print("SUMMARY")
print("="*60)

if problematic_images:
    print(f"⚠️  Issues found: {len(problematic_images)} bad images")
    print(f"✅ Clean dataset created with {copied} images")
    print(f"\n🚀 NEXT STEPS:")
    print(f"   1. Use the clean dataset: {clean_dataset}")
    print(f"   2. Update data_yaml path in training code")
    print(f"   3. Re-run training")
else:
    print("✅ No issues found!")
    print("\n🤔 If you still get errors, try:")
    print("   1. Add workers=0 to training")
    print("   2. Add cache=False to training")
    print("   3. Reduce batch size to 8")

print("="*60)

DATASET DIAGNOSTIC TOOL

🔍 Scanning dataset: /kaggle/input/speedlimit
✓ Found 4044 images

🧪 Testing image loading...
   Tested 4044/4044 images    

📊 DIAGNOSTIC RESULTS:
✅ All images are OK!


🔍 CHECKING FOR SEGMENT ISSUES:
✅ No segmentation issues detected


SUMMARY
✅ No issues found!

🤔 If you still get errors, try:
   1. Add workers=0 to training
   2. Add cache=False to training
   3. Reduce batch size to 8


In [2]:
print("📦 Installing packages...")
!pip install -q ultralytics opencv-python-headless PyYAML
print("✓ Done\n")

import os
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import display, Image as IPImage
import torch
import shutil

print(f"✓ GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   {torch.cuda.get_device_name(0)}\n")

📦 Installing packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 38.5 MB/s eta 0:00:0000:01m0:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━

In [3]:
datasets = os.listdir('/kaggle/input')
DATASET = f"/kaggle/input/{datasets[0]}"

# Output directory - GUARANTEED LOCATION
OUTPUT_DIR = "/kaggle/working/outputs"
MODEL_DIR = f"{OUTPUT_DIR}/model"
os.makedirs(MODEL_DIR, exist_ok=True)

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Dataset: {DATASET}")
print(f"Output Dir: {OUTPUT_DIR}")
print(f"Model Dir: {MODEL_DIR}")

# Training config
EPOCHS = 100
BATCH = 16
IMG_SIZE = 640
PATIENCE = 20

print(f"\nEpochs: {EPOCHS} | Batch: {BATCH} | Size: {IMG_SIZE}")

CONFIGURATION
Dataset: /kaggle/input/speedlimit
Output Dir: /kaggle/working/outputs
Model Dir: /kaggle/working/outputs/model

Epochs: 100 | Batch: 16 | Size: 640


In [4]:
print("\n" + "="*60)
print("DATASET ANALYSIS")
print("="*60)

# Find images
train_imgs = list(Path(DATASET).rglob('**/train/**/*.jpg')) + \
             list(Path(DATASET).rglob('**/train/**/*.png'))
val_imgs = list(Path(DATASET).rglob('**/val*/**/*.jpg')) + \
           list(Path(DATASET).rglob('**/val*/**/*.png'))

print(f"Train: {len(train_imgs)} images")
print(f"Val: {len(val_imgs)} images")

# Find or create data.yaml
yaml_files = list(Path(DATASET).rglob('*.yaml'))
if yaml_files:
    data_yaml = str(yaml_files[0])
    with open(data_yaml) as f:
        config = yaml.safe_load(f)
        classes = config.get('names', [])
    print(f"✓ Found data.yaml")
else:
    # Create data.yaml
    classes = ['speed_limit_30', 'speed_limit_50', 'speed_limit_70', 
               'speed_limit_90', 'stop']
    config = {
        'path': DATASET,
        'train': 'train/images',
        'val': 'valid/images',
        'nc': len(classes),
        'names': classes
    }
    data_yaml = f"{OUTPUT_DIR}/data.yaml"
    with open(data_yaml, 'w') as f:
        yaml.dump(config, f)
    print(f"✓ Created data.yaml")

print(f"Classes: {classes}")


DATASET ANALYSIS
Train: 2743 images
Val: 678 images
✓ Found data.yaml
Classes: ['speed_limit_100', 'speed_limit_120', 'speed_limit_20', 'speed_limit_30', 'speed_limit_40', 'speed_limit_50', 'speed_limit_60', 'speed_limit_70', 'speed_limit_80', 'speed_limit_90', 'stop']


In [ ]:
print("\n" + "🎯"*30)
print("TRAINING")
print("🎯"*30)

try:
    model = YOLO('yolov8n.pt')
    
    print(f"\n🚀 Starting training...")
    print(f"   Output will be saved to: {MODEL_DIR}")
    print("="*60)
    
    results = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=OUTPUT_DIR,
        name='training',
        patience=PATIENCE,
        save=True,
        device=0,
        plots=True,
        exist_ok=True,
        
        # Fixes for common errors
        amp=False,
        workers=0,
        cache=False,
    )
    
    # Copy best model to guaranteed location
    trained_model = f"{OUTPUT_DIR}/training/weights/best.pt"
    if os.path.exists(trained_model):
        shutil.copy2(trained_model, f"{MODEL_DIR}/best.pt")
        shutil.copy2(f"{OUTPUT_DIR}/training/weights/last.pt", f"{MODEL_DIR}/last.pt")
        print("\n" + "="*60)
        print("✅ TRAINING COMPLETE!")
        print("="*60)
        print(f"✓ Model saved to: {MODEL_DIR}/best.pt")
    else:
        print("\n⚠️  Warning: Model not found at expected location")
    
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    raise


🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯
TRAINING
🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯

🚀 Starting training...
   Output will be saved to: /kaggle/working/outputs/model
Ultralytics 8.3.229 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/speedlimit/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, mult

In [ ]:
print("\n" + "="*60)
print("VALIDATION")
print("="*60)

model_path = f"{MODEL_DIR}/best.pt"

if os.path.exists(model_path):
    model = YOLO(model_path)
    metrics = model.val(data=data_yaml)
    
    print(f"\n📊 RESULTS:")
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   mAP50:    {metrics.box.map50:.4f}")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall:    {metrics.box.mr:.4f}")
    
    # Show plots
    results_plot = f"{OUTPUT_DIR}/training/results.png"
    if os.path.exists(results_plot):
        shutil.copy2(results_plot, f"{OUTPUT_DIR}/results.png")
        print("\n📈 Training curves:")
        display(IPImage(filename=results_plot))
    
    confusion_plot = f"{OUTPUT_DIR}/training/confusion_matrix.png"
    if os.path.exists(confusion_plot):
        shutil.copy2(confusion_plot, f"{OUTPUT_DIR}/confusion_matrix.png")
        print("\n📈 Confusion matrix:")
        display(IPImage(filename=confusion_plot))
else:
    print(f"❌ Model not found at {model_path}")

In [ ]:
print("\n" + "="*60)
print("TESTING")
print("="*60)

if val_imgs and os.path.exists(model_path):
    samples = np.random.choice(val_imgs, min(6, len(val_imgs)), replace=False)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    
    for i, img_path in enumerate(samples):
        results = model(str(img_path), conf=0.25)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(annotated_rgb)
        axes[i].set_title(f"Test {i+1}", fontsize=10)
        axes[i].axis('off')
    
    plt.suptitle("Predictions", fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/predictions.png', dpi=100)
    plt.show()
    print(f"✓ Saved predictions to: {OUTPUT_DIR}/predictions.png")

In [ ]:
print("\n" + "="*60)
print("EXPORT")
print("="*60)

if os.path.exists(model_path):
    try:
        model = YOLO(model_path)
        
        onnx_path = model.export(format='onnx')
        if os.path.exists(onnx_path):
            shutil.copy2(onnx_path, f"{MODEL_DIR}/model.onnx")
            print(f"✓ ONNX: {MODEL_DIR}/model.onnx")
        
        print("✓ Export complete")
    except Exception as e:
        print(f"⚠️  Export warning: {e}")

In [ ]:
print("\n" + "="*60)
print("🎉 COMPLETE - OUTPUT VERIFICATION")
print("="*60)

# Verify all outputs
outputs = {
    'Model (best.pt)': f"{MODEL_DIR}/best.pt",
    'Model (last.pt)': f"{MODEL_DIR}/last.pt",
    'ONNX Export': f"{MODEL_DIR}/model.onnx",
    'Training Curves': f"{OUTPUT_DIR}/results.png",
    'Confusion Matrix': f"{OUTPUT_DIR}/confusion_matrix.png",
    'Predictions': f"{OUTPUT_DIR}/predictions.png",
}

print("\n📁 OUTPUT FILES:")
all_exist = True
for name, path in outputs.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"   ✅ {name}")
        print(f"      {path} ({size:.2f} MB)")
    else:
        print(f"   ❌ {name} - NOT FOUND")
        all_exist = False

if all_exist:
    print("\n✅ ALL FILES SAVED SUCCESSFULLY!")
else:
    print("\n⚠️  Some files missing - check errors above")

# Create download package
print("\n📦 Creating download package...")
zip_path = '/kaggle/working/speed_limit_model'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"✓ Created: {zip_path}.zip")

# Summary
summary = f"""
TRAINING SUMMARY
{'='*60}

Dataset: {len(train_imgs)} train, {len(val_imgs)} val images
Model: YOLOv8-Nano
Epochs: {EPOCHS}
Classes: {classes}

Performance:
- mAP50-95: {metrics.box.map:.4f}
- mAP50: {metrics.box.map50:.4f}
- Precision: {metrics.box.mp:.4f}
- Recall: {metrics.box.mr:.4f}

Files Location: {OUTPUT_DIR}
Main Model: {MODEL_DIR}/best.pt
Download Package: {zip_path}.zip
"""

with open(f'{OUTPUT_DIR}/SUMMARY.txt', 'w') as f:
    f.write(summary)

print(summary)

print("\n📥 HOW TO DOWNLOAD:")
print("="*60)
print("METHOD 1: Individual files")
print("   1. Left sidebar → 📁 folder icon")
print("   2. Navigate to: working/outputs/model/")
print("   3. Right-click best.pt → Download")
print("\nMETHOD 2: Complete package")
print("   1. Right sidebar → Output tab")
print("   2. Download: speed_limit_model.zip")
print("\nMETHOD 3: Use this path directly:")
print(f"   {MODEL_DIR}/best.pt")
print("="*60)

# List everything in output directory
print("\n📂 Complete output directory listing:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / (1024 * 1024)
        print(f'{subindent}📄 {file} ({size:.2f} MB)')

print("\n✨ TRAINING COMPLETE!")
print(f"✨ Your model is ready at: {MODEL_DIR}/best.pt")
print("="*60)

In [ ]:
print("\n" + "="*60)       
print("VALIDATION")                                              
print("="*60)

try:
    best_path = '/kaggle/working/runs/detect/speed_limit_detection/weights/best.pt'
    print(f"\n📥 Loading: {best_path}")
    model = YOLO(best_path)
    
    print("\n🔍 Validating...")
    metrics = model.val(data=data_yaml)
    
    print(f"\n📊 METRICS:")
    print("="*60)
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   mAP50:    {metrics.box.map50:.4f}")
    print(f"   mAP75:    {metrics.box.map75:.4f}")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall:    {metrics.box.mr:.4f}")
    print("="*60)
    
    # Show confusion matrix
    cm_path = Path(best_path).parent.parent / 'confusion_matrix.png'
    if cm_path.exists():
        print("\n📈 Confusion Matrix:")
        display(IPImage(filename=str(cm_path)))
    
    # Show results
    res_path = Path(best_path).parent.parent / 'results.png'
    if res_path.exists():
        print("\n📈 Training Curves:")
        display(IPImage(filename=str(res_path)))
    
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
print("\n" + "="*60)
print("INFERENCE TEST")
print("="*60)

try:
    n_test = 6
    
    if not dataset_info['val_img']:
        print("⚠️  No validation images")
    else:
        imgs = dataset_info['val_img']
        samples = np.random.choice(imgs, min(n_test, len(imgs)), replace=False)
        
        print(f"\n🔮 Testing on {len(samples)} images...\n")
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.ravel()
        
        for i, img_path in enumerate(samples):
            results = model(str(img_path), conf=0.25)
            annotated = results[0].plot()
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            
            axes[i].imshow(annotated_rgb)
            axes[i].set_title(f"Test {i+1}", fontsize=10)
            axes[i].axis('off')
            
            print(f"Image {i+1}: {img_path.name}")
            if len(results[0].boxes) == 0:
                print("   - No detections")
            else:
                for box in results[0].boxes:
                    cls = int(box.cls[0])
                    conf = float(box.conf[0])
                    name = model.names[cls]
                    print(f"   - {name}: {conf:.3f}")
            print()
        
        plt.suptitle("Inference Results", fontsize=14, weight='bold')
        plt.tight_layout()
        plt.savefig(f'{cfg.WORK_DIR}/inference.png', dpi=100)
        plt.show()
        
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
print("\n" + "="*60)
print("EXPORTING MODEL")
print("="*60)

try:
    model = YOLO(best_path)
    
    print("\n📦 Exporting to ONNX...")
    onnx = model.export(format='onnx')
    print(f"   ✓ {onnx}")
    
    print("\n📦 Exporting to TorchScript...")
    ts = model.export(format='torchscript')
    print(f"   ✓ {ts}")
    
    print("\n✅ Export complete!")
    
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)

print("\n📁 MODEL LOCATION:")
print("   /kaggle/working/runs/detect/speed_limit_detection/weights/")
print("   ├── best.pt  ← USE THIS")
print("   └── last.pt")

print("\n📊 PERFORMANCE:")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   mAP50:    {metrics.box.map50:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall:    {metrics.box.mr:.4f}")

print("\n📥 HOW TO DOWNLOAD:")
print("="*60)
print("""
1. Click folder icon (📁) on left
2. Go to: runs/detect/speed_limit_detection/weights/
3. Right-click 'best.pt' → Download
4. Also download:
   - *.onnx (for TensorFlow Lite)
   - *.torchscript (for PyTorch)
""")

print("\n🚀 NEXT STEPS:")
print("="*60)
print("""
1. ✅ Download trained model (best.pt)
2. ✅ Test on new images
3. ✅ Convert ONNX to TFLite for mobile
4. ✅ Implement OpenCV post-processing
5. ⏳ Train pothole detection next
""")

print("\n✨ SUCCESS! Model ready for deployment!")
print("="*60)

# Save summary
summary = f"""
SPEED LIMIT DETECTION - TRAINING SUMMARY
{'='*60}

Configuration:
- Model: YOLOv8-Nano
- Epochs: {cfg.EPOCHS}
- Batch: {cfg.BATCH}
- Image Size: {cfg.IMG_SIZE}
- Classes: {dataset_info['classes']}

Dataset:
- Train: {len(dataset_info['train_img'])} images
- Val: {len(dataset_info['val_img'])} images

Performance:
- mAP50-95: {metrics.box.map:.4f}
- mAP50: {metrics.box.map50:.4f}
- Precision: {metrics.box.mp:.4f}
- Recall: {metrics.box.mr:.4f}

Model: /kaggle/working/runs/detect/speed_limit_detection/weights/best.pt
"""

with open(f'{cfg.WORK_DIR}/summary.txt', 'w') as f:
    f.write(summary)

print("\n📄 Summary saved to: summary.txt")